# Лабораторная работа: RAG-система для Конституции Российской Федерации

**Цель:** Построение полноценной RAG (Retrieval-Augmented Generation) системы для поиска правовых статей из Конституции РФ с web-интерфейсом.

---

## Технологический стек (Этап 1)

| Компонент | Технология | Обоснование |
|-----------|-----------|-------------|
| **Эмбеддинги** | `sentence-transformers` (`paraphrase-multilingual-MiniLM-L12-v2`) | Бесплатная, поддерживает русский язык, работает локально |
| **LLM** | Anthropic Claude API (`claude-sonnet-4-20250514`) | Высокое качество ответов, поддержка русского языка |
| **Векторная БД** | `ChromaDB` | Open source, встроенная, не требует отдельного сервера |
| **Реранкер** | `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` | Многоязычный cross-encoder от sentence-transformers |
| **API** | `FastAPI` | Современный, быстрый, с автодокументацией |
| **Web-интерфейс** | `Gradio` | Простой, встраивается в ноутбук, подходит для прототипа |
| **Фреймворк** | `LangChain` | Стандарт отрасли для RAG-пайплайнов |

---
## Установка зависимостей

In [1]:
%%capture
!pip install sentence-transformers chromadb langchain langchain-community \
             langchain-anthropic anthropic fastapi uvicorn gradio \
             python-docx requests nest-asyncio httpx

---
## Этап 2: Работа с базой знаний

### 2.1 Загрузка и анализ структуры документа

In [2]:
import re
import json
from pathlib import Path

# Текст Конституции РФ
# для работы без скачивания файла
CONSTITUTION_TEXT = """
КОНСТИТУЦИЯ РОССИЙСКОЙ ФЕДЕРАЦИИ
Принята всенародным голосованием 12 декабря 1993 года

ПРЕАМБУЛА
Мы, многонациональный народ Российской Федерации, соединённые общей судьбой на своей земле, утверждая права и свободы человека, гражданский мир и согласие, сохраняя исторически сложившееся государственное единство, исходя из общепризнанных принципов равноправия и самоопределения народов, чтя память предков, передавших нам любовь и уважение к Отечеству, веру в добро и справедливость, возрождая суверенную государственность России и утверждая незыблемость её демократической основы, стремясь обеспечить благополучие и процветание России, исходя из ответственности за свою Родину перед нынешним и будущими поколениями, сознавая себя частью мирового сообщества, принимаем КОНСТИТУЦИЮ РОССИЙСКОЙ ФЕДЕРАЦИИ.

РАЗДЕЛ ПЕРВЫЙ

ГЛАВА 1. ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ

Статья 1
1. Российская Федерация - Россия есть демократическое федеративное правовое государство с республиканской формой правления.
2. Наименования Российская Федерация и Россия равнозначны.

Статья 2
Человек, его права и свободы являются высшей ценностью. Признание, соблюдение и защита прав и свобод человека и гражданина - обязанность государства.

Статья 3
1. Носителем суверенитета и единственным источником власти в Российской Федерации является её многонациональный народ.
2. Народ осуществляет свою власть непосредственно, а также через органы государственной власти и органы местного самоуправления.
3. Высшим непосредственным выражением власти народа являются референдум и свободные выборы.
4. Никто не может присваивать власть в Российской Федерации. Захват власти или присвоение властных полномочий преследуются по федеральному закону.

Статья 4
1. Суверенитет Российской Федерации распространяется на всю её территорию.
2. Конституция Российской Федерации и федеральные законы имеют верховенство на всей территории Российской Федерации.
3. Российская Федерация обеспечивает целостность и неприкосновенность своей территории.

Статья 5
1. Российская Федерация состоит из республик, краёв, областей, городов федерального значения, автономной области, автономных округов - равноправных субъектов Российской Федерации.
2. Республика (государство) имеет свою конституцию и законодательство. Край, область, город федерального значения, автономная область, автономный округ имеет свой устав и законодательство.
3. Федеративное устройство Российской Федерации основано на её государственной целостности, единстве системы государственной власти, разграничении предметов ведения и полномочий между органами государственной власти Российской Федерации и органами государственной власти субъектов Российской Федерации, равноправии и самоопределении народов в Российской Федерации.
4. Во взаимоотношениях с федеральными органами государственной власти все субъекты Российской Федерации между собой равноправны.

Статья 6
1. Гражданство Российской Федерации приобретается и прекращается в соответствии с федеральным законом, является единым и равным независимо от оснований приобретения.
2. Каждый гражданин Российской Федерации обладает на её территории всеми правами и свободами и несёт равные обязанности, предусмотренные Конституцией Российской Федерации.
3. Гражданин Российской Федерации не может быть лишён своего гражданства или права изменить его.

Статья 7
1. Российская Федерация - социальное государство, политика которого направлена на создание условий, обеспечивающих достойную жизнь и свободное развитие человека.
2. В Российской Федерации охраняются труд и здоровье людей, устанавливается гарантированный минимальный размер оплаты труда, обеспечивается государственная поддержка семьи, материнства, отцовства и детства, инвалидов и пожилых граждан, развивается система социальных служб, устанавливаются государственные пенсии, пособия и иные гарантии социальной защиты.

Статья 8
1. В Российской Федерации гарантируются единство экономического пространства, свободное перемещение товаров, услуг и финансовых средств, поддержка конкуренции, свобода экономической деятельности.
2. В Российской Федерации признаются и защищаются равным образом частная, государственная, муниципальная и иные формы собственности.

Статья 9
1. Земля и другие природные ресурсы используются и охраняются в Российской Федерации как основа жизни и деятельности народов, проживающих на соответствующей территории.
2. Земля и другие природные ресурсы могут находиться в частной, государственной, муниципальной и иных формах собственности.

Статья 10
Государственная власть в Российской Федерации осуществляется на основе разделения на законодательную, исполнительную и судебную. Органы законодательной, исполнительной и судебной власти самостоятельны.

Статья 11
1. Государственную власть в Российской Федерации осуществляют Президент Российской Федерации, Федеральное Собрание (Совет Федерации и Государственная Дума), Правительство Российской Федерации, суды Российской Федерации.
2. Государственную власть в субъектах Российской Федерации осуществляют образуемые ими органы государственной власти.
3. Разграничение предметов ведения и полномочий между органами государственной власти Российской Федерации и органами государственной власти субъектов Российской Федерации осуществляется настоящей Конституцией, Федеративным и иными договорами о разграничении предметов ведения и полномочий.

Статья 12
В Российской Федерации признаётся и гарантируется местное самоуправление. Местное самоуправление в пределах своих полномочий самостоятельно. Органы местного самоуправления не входят в систему органов государственной власти.

Статья 13
1. В Российской Федерации признаётся идеологическое многообразие.
2. Никакая идеология не может устанавливаться в качестве государственной или обязательной.
3. В Российской Федерации признаются политическое многообразие, многопартийность.
4. Общественные объединения равны перед законом.
5. Запрещается создание и деятельность общественных объединений, цели или действия которых направлены на насильственное изменение основ конституционного строя и нарушение целостности Российской Федерации, подрыв безопасности государства, создание вооружённых формирований, разжигание социальной, расовой, национальной и религиозной розни.

Статья 14
1. Российская Федерация - светское государство. Никакая религия не может устанавливаться в качестве государственной или обязательной.
2. Религиозные объединения отделены от государства и равны перед законом.

Статья 15
1. Конституция Российской Федерации имеет высшую юридическую силу, прямое действие и применяется на всей территории Российской Федерации. Законы и иные правовые акты, принимаемые в Российской Федерации, не должны противоречить Конституции Российской Федерации.
2. Органы государственной власти, органы местного самоуправления, должностные лица, граждане и их объединения обязаны соблюдать Конституцию Российской Федерации и законы.
3. Законы подлежат официальному опубликованию. Неопубликованные законы не применяются. Любые нормативные правовые акты, затрагивающие права, свободы и обязанности человека и гражданина, не могут применяться, если они не опубликованы официально для всеобщего сведения.
4. Общепризнанные принципы и нормы международного права и международные договоры Российской Федерации являются составной частью её правовой системы. Если международным договором Российской Федерации установлены иные правила, чем предусмотренные законом, то применяются правила международного договора.

Статья 16
1. Положения настоящей главы Конституции составляют основы конституционного строя Российской Федерации и не могут быть изменены иначе как в порядке, установленном настоящей Конституцией.
2. Никакие другие положения настоящей Конституции не могут противоречить основам конституционного строя Российской Федерации.

ГЛАВА 2. ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА

Статья 17
1. В Российской Федерации признаются и гарантируются права и свободы человека и гражданина согласно общепризнанным принципам и нормам международного права и в соответствии с настоящей Конституцией.
2. Основные права и свободы человека неотчуждаемы и принадлежат каждому от рождения.
3. Осуществление прав и свобод человека и гражданина не должно нарушать права и свободы других лиц.

Статья 18
Права и свободы человека и гражданина являются непосредственно действующими. Они определяют смысл, содержание и применение законов, деятельность законодательной и исполнительной власти, местного самоуправления и обеспечиваются правосудием.

Статья 19
1. Все равны перед законом и судом.
2. Государство гарантирует равенство прав и свобод человека и гражданина независимо от пола, расы, национальности, языка, происхождения, имущественного и должностного положения, места жительства, отношения к религии, убеждений, принадлежности к общественным объединениям, а также других обстоятельств. Запрещаются любые формы ограничения прав граждан по признакам социальной, расовой, национальной, языковой или религиозной принадлежности.
3. Мужчина и женщина имеют равные права и свободы и равные возможности для их реализации.

Статья 20
1. Каждый имеет право на жизнь.
2. Смертная казнь впредь до её отмены может устанавливаться федеральным законом в качестве исключительной меры наказания за особо тяжкие преступления против жизни при предоставлении обвиняемому права на рассмотрение его дела судом с участием присяжных заседателей.

Статья 21
1. Достоинство личности охраняется государством. Ничто не может быть основанием для его умаления.
2. Никто не должен подвергаться пыткам, насилию, другому жестокому или унижающему человеческое достоинство обращению или наказанию. Никто не может быть без добровольного согласия подвергнут медицинским, научным или иным опытам.

Статья 22
1. Каждый имеет право на свободу и личную неприкосновенность.
2. Арест, заключение под стражу и содержание под стражей допускаются только по судебному решению. До судебного решения лицо не может быть подвергнуто задержанию на срок более 48 часов.

Статья 23
1. Каждый имеет право на неприкосновенность частной жизни, личную и семейную тайну, защиту своей чести и доброго имени.
2. Каждый имеет право на тайну переписки, телефонных переговоров, почтовых, телеграфных и иных сообщений. Ограничение этого права допускается только на основании судебного решения.

Статья 24
1. Сбор, хранение, использование и распространение информации о частной жизни лица без его согласия не допускаются.
2. Органы государственной власти и органы местного самоуправления, их должностные лица обязаны обеспечить каждому возможность ознакомления с документами и материалами, непосредственно затрагивающими его права и свободы, если иное не предусмотрено законом.

Статья 25
Жилище неприкосновенно. Никто не вправе проникать в жилище против воли проживающих в нём лиц иначе как в случаях, установленных федеральным законом, или на основании судебного решения.

Статья 26
1. Каждый вправе определять и указывать свою национальную принадлежность. Никто не может быть принуждён к определению и указанию своей национальной принадлежности.
2. Каждый имеет право на пользование родным языком, на свободный выбор языка общения, воспитания, обучения и творчества.

Статья 27
1. Каждый, кто законно находится на территории Российской Федерации, имеет право свободно передвигаться, выбирать место пребывания и жительства.
2. Каждый может свободно выезжать за пределы Российской Федерации. Гражданин Российской Федерации имеет право беспрепятственно возвращаться в Российскую Федерацию.

Статья 28
Каждому гарантируется свобода совести, свобода вероисповедания, включая право исповедовать индивидуально или совместно с другими любую религию или не исповедовать никакой, совершать религиозные обряды и иные действия.

Статья 29
1. Каждому гарантируется свобода мысли и слова.
2. Не допускаются пропаганда или агитация, возбуждающие социальную, расовую, национальную или религиозную ненависть и вражду. Запрещается пропаганда социального, расового, национального, религиозного или языкового превосходства.
3. Никто не может быть принуждён к выражению своих мнений и убеждений или отказу от них.
4. Каждый имеет право свободно искать, получать, передавать, производить и распространять информацию любым законным способом. Перечень сведений, составляющих государственную тайну, определяется федеральным законом.
5. Гарантируется свобода массовой информации. Цензура запрещается.

Статья 30
1. Каждый имеет право на объединение, включая право создавать профессиональные союзы для защиты своих интересов. Свобода деятельности общественных объединений гарантируется.
2. Никто не может быть принуждён к вступлению в какое-либо объединение или пребыванию в нём.

Статья 31
Граждане Российской Федерации имеют право собираться мирно, без оружия, проводить собрания, митинги и демонстрации, шествия и пикетирование.

Статья 32
1. Граждане Российской Федерации имеют право участвовать в управлении делами государства как непосредственно, так и через своих представителей.
2. Граждане Российской Федерации имеют право избирать и быть избранными в органы государственной власти и органы местного самоуправления, а также участвовать в референдуме.
3. Не имеют права избирать и быть избранными граждане, признанные судом недееспособными, а также содержащиеся в местах лишения свободы по приговору суда.
4. Граждане Российской Федерации имеют равный доступ к государственной службе.
5. Граждане Российской Федерации имеют право участвовать в отправлении правосудия.

Статья 33
Граждане Российской Федерации имеют право обращаться лично, а также направлять индивидуальные и коллективные обращения в государственные органы и органы местного самоуправления.

Статья 34
1. Каждый имеет право на свободное использование своих способностей и имущества для предпринимательской и иной не запрещённой законом экономической деятельности.
2. Не допускается экономическая деятельность, направленная на монополизацию и недобросовестную конкуренцию.

Статья 35
1. Право частной собственности охраняется законом.
2. Каждый вправе иметь имущество в собственности, владеть, пользоваться и распоряжаться им как единолично, так и совместно с другими лицами.
3. Никто не может быть лишён своего имущества иначе как по решению суда. Принудительное отчуждение имущества для государственных нужд может быть произведено только при условии предварительного и равноценного возмещения.
4. Право наследования гарантируется.

Статья 36
1. Граждане и их объединения вправе иметь в частной собственности землю.
2. Владение, пользование и распоряжение землёй и другими природными ресурсами осуществляются их собственниками свободно, если это не наносит ущерба окружающей среде и не нарушает прав и законных интересов иных лиц.
3. Условия и порядок пользования землёй определяются на основе федерального закона.

Статья 37
1. Труд свободен. Каждый имеет право свободно распоряжаться своими способностями к труду, выбирать род деятельности и профессию.
2. Принудительный труд запрещён.
3. Каждый имеет право на труд в условиях, отвечающих требованиям безопасности и гигиены, на вознаграждение за труд без какой бы то ни было дискриминации и не ниже установленного федеральным законом минимального размера оплаты труда, а также право на защиту от безработицы.
4. Признаётся право на индивидуальные и коллективные трудовые споры с использованием установленных федеральным законом способов их разрешения, включая право на забастовку.
5. Каждый имеет право на отдых. Работающему по трудовому договору гарантируются установленные федеральным законом продолжительность рабочего времени, выходные и праздничные дни, оплачиваемый ежегодный отпуск.

Статья 38
1. Материнство и детство, семья находятся под защитой государства.
2. Забота о детях, их воспитание - равное право и обязанность родителей.
3. Трудоспособные дети, достигшие 18 лет, должны заботиться о нетрудоспособных родителях.

Статья 39
1. Каждому гарантируется социальное обеспечение по возрасту, в случае болезни, инвалидности, потери кормильца, для воспитания детей и в иных случаях, установленных законом.
2. Государственные пенсии и социальные пособия устанавливаются законом.
3. Поощряются добровольное социальное страхование, создание дополнительных форм социального обеспечения и благотворительность.

Статья 40
1. Каждый имеет право на жилище. Никто не может быть произвольно лишён жилища.
2. Органы государственной власти и органы местного самоуправления поощряют жилищное строительство, создают условия для осуществления права на жилище.
3. Малоимущим, иным указанным в законе гражданам, нуждающимся в жилище, оно предоставляется бесплатно или за доступную плату из государственных, муниципальных и других жилищных фондов в соответствии с установленными законом нормами.

Статья 41
1. Каждый имеет право на охрану здоровья и медицинскую помощь. Медицинская помощь в государственных и муниципальных учреждениях здравоохранения оказывается гражданам бесплатно за счёт средств соответствующего бюджета, страховых взносов, других поступлений.
2. В Российской Федерации финансируются федеральные программы охраны и укрепления здоровья населения, принимаются меры по развитию государственной, муниципальной, частной систем здравоохранения, поощряется деятельность, способствующая укреплению здоровья человека, развитию физической культуры и спорта, экологическому и санитарно-эпидемиологическому благополучию.
3. Сокрытие должностными лицами фактов и обстоятельств, создающих угрозу для жизни и здоровья людей, влечёт за собой ответственность в соответствии с федеральным законом.

Статья 42
Каждый имеет право на благоприятную окружающую среду, достоверную информацию о её состоянии и на возмещение ущерба, причинённого его здоровью или имуществу экологическим правонарушением.

Статья 43
1. Каждый имеет право на образование.
2. Гарантируются общедоступность и бесплатность дошкольного, основного общего и среднего профессионального образования в государственных или муниципальных образовательных учреждениях и на предприятиях.
3. Каждый вправе на конкурсной основе бесплатно получить высшее образование в государственном или муниципальном образовательном учреждении и на предприятии.
4. Основное общее образование обязательно. Родители или лица, их заменяющие, обеспечивают получение детьми основного общего образования.
5. Российская Федерация устанавливает федеральные государственные образовательные стандарты, поддерживает различные формы образования и самообразования.

Статья 44
1. Каждому гарантируется свобода литературного, художественного, научного, технического и других видов творчества, преподавания. Интеллектуальная собственность охраняется законом.
2. Каждый имеет право на участие в культурной жизни и пользование учреждениями культуры, на доступ к культурным ценностям.
3. Каждый обязан заботиться о сохранении исторического и культурного наследия, беречь памятники истории и культуры.

Статья 45
1. Государственная защита прав и свобод человека и гражданина в Российской Федерации гарантируется.
2. Каждый вправе защищать свои права и свободы всеми способами, не запрещёнными законом.

Статья 46
1. Каждому гарантируется судебная защита его прав и свобод.
2. Решения и действия (или бездействие) органов государственной власти, органов местного самоуправления, общественных объединений и должностных лиц могут быть обжалованы в суд.
3. Каждый вправе в соответствии с международными договорами Российской Федерации обращаться в межгосударственные органы по защите прав и свобод человека, если исчерпаны все имеющиеся внутригосударственные средства правовой защиты.

Статья 47
1. Никто не может быть лишён права на рассмотрение его дела в том суде и тем судьёй, к подсудности которых оно отнесено законом.
2. Обвиняемый в совершении преступления имеет право на рассмотрение его дела судом с участием присяжных заседателей в случаях, предусмотренных федеральным законом.

Статья 48
1. Каждому гарантируется право на получение квалифицированной юридической помощи. В случаях, предусмотренных законом, юридическая помощь оказывается бесплатно.
2. Каждый задержанный, заключённый под стражу, обвиняемый в совершении преступления имеет право пользоваться помощью адвоката (защитника) с момента соответственно задержания, заключения под стражу или предъявления обвинения.

Статья 49
1. Каждый обвиняемый в совершении преступления считается невиновным, пока его виновность не будет доказана в предусмотренном федеральным законом порядке и установлена вступившим в законную силу приговором суда.
2. Обвиняемый не обязан доказывать свою невиновность.
3. Неустранимые сомнения в виновности лица толкуются в пользу обвиняемого.

Статья 50
1. Никто не может быть повторно осуждён за одно и то же преступление.
2. При осуществлении правосудия не допускается использование доказательств, полученных с нарушением федерального закона.
3. Каждый осуждённый за преступление имеет право на пересмотр приговора вышестоящим судом в порядке, установленном федеральным законом, а также право просить о помиловании или смягчении наказания.

Статья 51
1. Никто не обязан свидетельствовать против себя самого, своего супруга и близких родственников, круг которых определяется федеральным законом.
2. Федеральным законом могут устанавливаться иные случаи освобождения от обязанности давать свидетельские показания.

Статья 52
Права потерпевших от преступлений и злоупотреблений властью охраняются законом. Государство обеспечивает потерпевшим доступ к правосудию и компенсацию причинённого ущерба.

Статья 53
Каждый имеет право на возмещение государством вреда, причинённого незаконными действиями (или бездействием) органов государственной власти или их должностных лиц.

Статья 54
1. Закон, устанавливающий или отягчающий ответственность, обратной силы не имеет.
2. Никто не может нести ответственность за деяние, которое в момент его совершения не признавалось правонарушением. Если после совершения правонарушения ответственность за него устранена или смягчена, применяется новый закон.

Статья 55
1. Перечисление в Конституции Российской Федерации основных прав и свобод не должно толковаться как отрицание или умаление других общепризнанных прав и свобод человека и гражданина.
2. В Российской Федерации не должны издаваться законы, отменяющие или умаляющие права и свободы человека и гражданина.
3. Права и свободы человека и гражданина могут быть ограничены федеральным законом только в той мере, в какой это необходимо в целях защиты основ конституционного строя, нравственности, здоровья, прав и законных интересов других лиц, обеспечения обороны страны и безопасности государства.

Статья 56
1. В условиях чрезвычайного положения для обеспечения безопасности граждан и защиты конституционного строя в соответствии с федеральным конституционным законом могут устанавливаться отдельные ограничения прав и свобод с указанием пределов и срока их действия.
2. Чрезвычайное положение может вводиться при наличии обстоятельств и в порядке, установленных федеральным конституционным законом.
3. Не подлежат ограничению права и свободы, предусмотренные статьями 20, 21, 23 (часть 1), 24, 25, 28, 34 (часть 1), 40 (часть 1), 46 - 54 Конституции Российской Федерации.

Статья 57
Каждый обязан платить законно установленные налоги и сборы. Законы, устанавливающие новые налоги или ухудшающие положение налогоплательщиков, обратной силы не имеют.

Статья 58
Каждый обязан сохранять природу и окружающую среду, бережно относиться к природным богатствам.

Статья 59
1. Защита Отечества является долгом и обязанностью гражданина Российской Федерации.
2. Гражданин Российской Федерации несёт военную службу в соответствии с федеральным законом.
3. Гражданин Российской Федерации в случае, если его убеждениям или вероисповеданию противоречит несение военной службы, а также в иных установленных федеральным законом случаях имеет право на замену её альтернативной гражданской службой.

Статья 60
Гражданин Российской Федерации может самостоятельно осуществлять в полном объёме свои права и обязанности с 18 лет.

Статья 61
1. Гражданин Российской Федерации не может быть выслан за пределы Российской Федерации или выдан другому государству.
2. Российская Федерация гарантирует своим гражданам защиту и покровительство за её пределами.

Статья 62
1. Гражданин Российской Федерации может иметь гражданство иностранного государства (двойное гражданство) в соответствии с федеральным законом или международным договором Российской Федерации.
2. Наличие у гражданина Российской Федерации гражданства иностранного государства не умаляет его прав и свобод и не освобождает от обязанностей, вытекающих из российского гражданства, если иное не предусмотрено федеральным законом или международным договором Российской Федерации.
3. Иностранные граждане и лица без гражданства пользуются в Российской Федерации правами и несут обязанности наравне с гражданами Российской Федерации, кроме случаев, установленных федеральным законом или международным договором Российской Федерации.

Статья 63
1. Российская Федерация предоставляет политическое убежище иностранным гражданам и лицам без гражданства в соответствии с общепризнанными нормами международного права.
2. В Российской Федерации не допускается выдача другим государствам лиц, преследуемых за политические убеждения, а также за действия (или бездействие), не признаваемые в Российской Федерации преступлением. Выдача лиц, обвиняемых в совершении преступления, а также передача осуждённых для отбывания наказания в других государствах осуществляются на основе федерального закона или международного договора Российской Федерации.

Статья 64
Положения настоящей главы составляют основы правового статуса личности в Российской Федерации и не могут быть изменены иначе как в порядке, установленном настоящей Конституцией.

ГЛАВА 3. ФЕДЕРАТИВНОЕ УСТРОЙСТВО

Статья 65
1. В составе Российской Федерации находятся субъекты Российской Федерации: Республика Адыгея (Адыгея), Республика Алтай, Республика Башкортостан, Республика Бурятия, Республика Дагестан, Республика Ингушетия, Кабардино-Балкарская Республика, Республика Калмыкия, Карачаево-Черкесская Республика, Республика Карелия, Республика Коми, Республика Крым, Республика Марий Эл, Республика Мордовия, Республика Саха (Якутия), Республика Северная Осетия - Алания, Республика Татарстан (Татарстан), Республика Тыва, Удмуртская Республика, Республика Хакасия, Чеченская Республика, Чувашская Республика - Чувашия; Алтайский край, Забайкальский край, Камчатский край, Краснодарский край, Красноярский край, Пермский край, Приморский край, Ставропольский край, Хабаровский край; Амурская область, Архангельская область, Астраханская область, Белгородская область, Брянская область, Владимирская область, Волгоградская область, Вологодская область, Воронежская область, Донецкая Народная Республика, Ивановская область, Иркутская область, Калининградская область, Калужская область, Кемеровская область - Кузбасс, Кировская область, Костромская область, Курганская область, Курская область, Ленинградская область, Липецкая область, Луганская Народная Республика, Магаданская область, Московская область, Мурманская область, Нижегородская область, Новгородская область, Новосибирская область, Омская область, Оренбургская область, Орловская область, Пензенская область, Псковская область, Ростовская область, Рязанская область, Самарская область, Саратовская область, Сахалинская область, Свердловская область, Смоленская область, Тамбовская область, Тверская область, Томская область, Тульская область, Тюменская область, Ульяновская область, Херсонская область, Челябинская область, Запорожская область, Ярославская область; Москва, Санкт-Петербург, Севастополь - города федерального значения; Еврейская автономная область; Ненецкий автономный округ,Ханты-Мансийский автономный округ - Югра, Чукотский автономный округ, Ямало-Ненецкий автономный округ.

ГЛАВА 4. ПРЕЗИДЕНТ РОССИЙСКОЙ ФЕДЕРАЦИИ

Статья 80
1. Президент Российской Федерации является главой государства.
2. Президент Российской Федерации является гарантом Конституции Российской Федерации, прав и свобод человека и гражданина. В установленном Конституцией Российской Федерации порядке он принимает меры по охране суверенитета Российской Федерации, её независимости и государственной целостности, обеспечивает согласованное функционирование и взаимодействие органов, входящих в единую систему публичной власти.
3. Президент Российской Федерации в соответствии с Конституцией Российской Федерации и федеральными законами определяет основные направления внутренней и внешней политики государства.
4. Президент Российской Федерации как глава государства представляет Российскую Федерацию внутри страны и в международных отношениях.

Статья 81
1. Президент Российской Федерации избирается сроком на шесть лет гражданами Российской Федерации на основе всеобщего равного и прямого избирательного права при тайном голосовании.
2. Президентом Российской Федерации может быть избран гражданин Российской Федерации не моложе 35 лет, постоянно проживающий в Российской Федерации не менее 25 лет, не имеющий и не имевший ранее гражданства иностранного государства либо вида на жительство или иного документа, подтверждающего право на постоянное проживание гражданина Российской Федерации на территории иностранного государства.
3. Одно и то же лицо не может занимать должность Президента Российской Федерации более двух сроков.
4. Порядок выборов Президента Российской Федерации определяется федеральным законом.

Статья 82
1. При вступлении в должность Президент Российской Федерации приносит народу следующую присягу: «Клянусь при осуществлении полномочий Президента Российской Федерации уважать и охранять права и свободы человека и гражданина, соблюдать и защищать Конституцию Российской Федерации, защищать суверенитет и независимость, безопасность и целостность государства, верно служить народу».
2. Присяга приносится в торжественной обстановке в присутствии сенаторов Российской Федерации, депутатов Государственной Думы и судей Конституционного Суда Российской Федерации.

ГЛАВА 5. ФЕДЕРАЛЬНОЕ СОБРАНИЕ

Статья 94
Федеральное Собрание - парламент Российской Федерации - является представительным и законодательным органом Российской Федерации.

Статья 95
1. Федеральное Собрание состоит из двух палат - Совета Федерации и Государственной Думы.
2. В Совет Федерации входят: по два представителя от каждого субъекта Российской Федерации - по одному от законодательного (представительного) и исполнительного органов государственной власти; представители Российской Федерации, назначаемые Президентом Российской Федерации, число которых составляет не более десяти процентов от числа членов Совета Федерации - представителей от законодательных (представительных) и исполнительных органов государственной власти субъектов Российской Федерации.
3. Государственная Дума состоит из 450 депутатов.

ГЛАВА 7. СУДЕБНАЯ ВЛАСТЬ И ПРОКУРАТУРА

Статья 118
1. Правосудие в Российской Федерации осуществляется только судом.
2. Судебная власть осуществляется посредством конституционного, гражданского, арбитражного, административного и уголовного судопроизводства.
3. Судебная система Российской Федерации устанавливается Конституцией Российской Федерации и федеральным конституционным законом. Создание чрезвычайных судов не допускается.

Статья 119
Судьями могут быть граждане Российской Федерации, достигшие 25 лет, имеющие высшее юридическое образование и стаж работы по юридической профессии не менее пяти лет. Федеральным законом могут быть установлены дополнительные требования к судьям судов Российской Федерации.

Статья 120
1. Судьи независимы и подчиняются только Конституции Российской Федерации и федеральному закону.
2. Суд, установив при рассмотрении дела несоответствие акта государственного или иного органа закону, принимает решение в соответствии с законом.

Статья 121
1. Судьи несменяемы.
2. Полномочия судьи могут быть прекращены или приостановлены не иначе как в порядке и по основаниям, установленным федеральным законом.

Статья 122
1. Судьи неприкосновенны.
2. Судья не может быть привлечён к уголовной ответственности иначе как в порядке, определяемом федеральным законом.

ГЛАВА 9. КОНСТИТУЦИОННЫЕ ПОПРАВКИ И ПЕРЕСМОТР КОНСТИТУЦИИ

Статья 134
Предложения о поправках и пересмотре положений Конституции Российской Федерации могут вносить Президент Российской Федерации, Совет Федерации, Государственная Дума, Правительство Российской Федерации, законодательные (представительные) органы субъектов Российской Федерации, а также группа численностью не менее одной пятой членов Совета Федерации или депутатов Государственной Думы.

Статья 135
1. Положения глав 1, 2 и 9 Конституции Российской Федерации не могут быть пересмотрены Федеральным Собранием.
2. Если предложение о пересмотре положений глав 1, 2 и 9 Конституции Российской Федерации будет поддержано тремя пятыми голосов от общего числа сенаторов Российской Федерации и депутатов Государственной Думы, то в соответствии с федеральным конституционным законом созывается Конституционное Собрание.
3. Конституционное Собрание либо подтверждает неизменность Конституции Российской Федерации, либо разрабатывает проект новой Конституции Российской Федерации, который принимается Конституционным Собранием двумя третями голосов от общего числа его членов или выносится на всенародное голосование. При проведении всенародного голосования Конституция Российской Федерации считается принятой, если за неё проголосовало более половины избирателей, принявших участие в голосовании, при условии, что в нём приняло участие более половины избирателей.

Статья 136
Поправки к главам 3 - 8 Конституции Российской Федерации принимаются в порядке, предусмотренном для принятия федерального конституционного закона, и вступают в силу после их одобрения органами законодательной власти не менее чем двух третей субъектов Российской Федерации.
"""

print(f"Длина текста: {len(CONSTITUTION_TEXT)} символов")
print(f"Количество строк: {len(CONSTITUTION_TEXT.splitlines())}")

articles = re.findall(r'Статья\s+(\d+)', CONSTITUTION_TEXT)
chapters = re.findall(r'ГЛАВА\s+(\d+)\.\s+([^\n]+)', CONSTITUTION_TEXT)

print(f"\nНайдено статей: {len(articles)}")
print(f"Статьи: {articles}")
print(f"\nНайдено глав: {len(chapters)}")
for num, title in chapters:
    print(f"  Глава {num}: {title}")

Длина текста: 33703 символов
Количество строк: 362

Найдено статей: 78
Статьи: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '80', '81', '82', '94', '95', '118', '119', '120', '121', '122', '134', '135', '136']

Найдено глав: 7
  Глава 1: ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ
  Глава 2: ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА
  Глава 3: ФЕДЕРАТИВНОЕ УСТРОЙСТВО
  Глава 4: ПРЕЗИДЕНТ РОССИЙСКОЙ ФЕДЕРАЦИИ
  Глава 5: ФЕДЕРАЛЬНОЕ СОБРАНИЕ
  Глава 7: СУДЕБНАЯ ВЛАСТЬ И ПРОКУРАТУРА
  Глава 9: КОНСТИТУЦИОННЫЕ ПОПРАВКИ И ПЕРЕСМОТР КОНСТИТУЦИИ


### 2.2 Чанкирование текста с сохранением метаинформации

In [3]:
import re
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class ArticleChunk:
    article_number: int
    chapter_number: Optional[int]
    chapter_title: Optional[str]
    section: Optional[str]
    text: str
    chunk_id: str = field(init=False)
    
    def __post_init__(self):
        self.chunk_id = f"article_{self.article_number}"


def parse_constitution(text: str) -> List[ArticleChunk]:
    chunks = []
    lines = text.strip().split('\n')
    
    current_section = None
    current_chapter_num = None
    current_chapter_title = None
    current_article_num = None
    current_article_lines = []
    
    section_pattern = re.compile(r'^РАЗДЕЛ\s+', re.IGNORECASE)
    chapter_pattern = re.compile(r'^ГЛАВА\s+(\d+)\.\s+(.+)')
    article_pattern = re.compile(r'^Статья\s+(\d+)\s*$')
    
    def save_article():
        if current_article_num and current_article_lines:
            text_body = '\n'.join(current_article_lines).strip()
            if text_body:
                chunks.append(ArticleChunk(
                    article_number=current_article_num,
                    chapter_number=current_chapter_num,
                    chapter_title=current_chapter_title,
                    section=current_section,
                    text=f"Статья {current_article_num}\n{text_body}"
                ))
    
    for line in lines:
        line_stripped = line.strip()
        if not line_stripped:
            continue
        
        if section_pattern.match(line_stripped):
            save_article()
            current_section = line_stripped
            current_article_num = None
            current_article_lines = []
            continue
        
        chapter_match = chapter_pattern.match(line_stripped)
        if chapter_match:
            save_article()
            current_chapter_num = int(chapter_match.group(1))
            current_chapter_title = chapter_match.group(2).strip()
            current_article_num = None
            current_article_lines = []
            continue
        
        article_match = article_pattern.match(line_stripped)
        if article_match:
            save_article()
            current_article_num = int(article_match.group(1))
            current_article_lines = []
            continue
        
        if current_article_num is not None:
            current_article_lines.append(line_stripped)
    
    save_article()
    return chunks


chunks = parse_constitution(CONSTITUTION_TEXT)

print(f"Всего чанков: {len(chunks)}\n")
print("=" * 60)
print("Пример первых 3 чанков:")
print("=" * 60)
for chunk in chunks[:3]:
    print("\n")
    print(f"   ID: {chunk.chunk_id}")
    print(f"   Статья: {chunk.article_number}")
    print(f"   Глава: {chunk.chapter_number} — {chunk.chapter_title}")
    print(f"   Раздел: {chunk.section}")
    print(f"   Текст ({len(chunk.text)} симв.): {chunk.text[:200]}...")
    print("-" * 60)

Всего чанков: 78

Пример первых 3 чанков:


   ID: article_1
   Статья: 1
   Глава: 1 — ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ
   Раздел: РАЗДЕЛ ПЕРВЫЙ
   Текст (191 симв.): Статья 1
1. Российская Федерация - Россия есть демократическое федеративное правовое государство с республиканской формой правления.
2. Наименования Российская Федерация и Россия равнозначны....
------------------------------------------------------------


   ID: article_2
   Статья: 2
   Глава: 1 — ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ
   Раздел: РАЗДЕЛ ПЕРВЫЙ
   Текст (158 симв.): Статья 2
Человек, его права и свободы являются высшей ценностью. Признание, соблюдение и защита прав и свобод человека и гражданина - обязанность государства....
------------------------------------------------------------


   ID: article_3
   Статья: 3
   Глава: 1 — ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ
   Раздел: РАЗДЕЛ ПЕРВЫЙ
   Текст (495 симв.): Статья 3
1. Носителем суверенитета и единственным источником власти в Российской Федерации является её многона

### 2.3 и 2.4 Векторизация чанков и сохранение в ChromaDB

In [4]:
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

print("Загрузка модели эмбеддингов...")
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"   Модель загружена: paraphrase-multilingual-MiniLM-L12-v2")
print(f"   Размерность эмбеддинга: {embedding_model.get_embedding_dimension()}")

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("constitution_rf")
except:
    pass

collection = chroma_client.create_collection(
    name="constitution_rf",
    metadata={"hnsw:space": "cosine"}
)

print("\nВекторизация чанков...")
texts = [c.text for c in chunks]
embeddings = embedding_model.encode(texts, batch_size=128, show_progress_bar=True)

collection.add(
    ids=[c.chunk_id for c in chunks],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[
        {
            "article_number": c.article_number,
            "chapter_number": c.chapter_number or 0,
            "chapter_title": c.chapter_title or "",
            "section": c.section or ""
        }
        for c in chunks
    ]
)

print(f"\nДобавлено {collection.count()} документов в ChromaDB")
print("\nПример метаданных первой записи:")
sample = collection.get(ids=["article_1"], include=["documents", "metadatas"])
print(f"  Метаданные: {sample['metadatas'][0]}")
print(f"  Текст: {sample['documents'][0][:100]}...")

c:\Users\IronSmail\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загрузка модели эмбеддингов...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4852.54it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Модель загружена: paraphrase-multilingual-MiniLM-L12-v2
   Размерность эмбеддинга: 384

Векторизация чанков...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.05s/it]


Добавлено 78 документов в ChromaDB

Пример метаданных первой записи:
  Метаданные: {'chapter_number': 1, 'section': 'РАЗДЕЛ ПЕРВЫЙ', 'chapter_title': 'ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ', 'article_number': 1}
  Текст: Статья 1
1. Российская Федерация - Россия есть демократическое федеративное правовое государство с р...


---
## Этап 3: Формирование запроса к LLM и промпт-инжиниринг

### 3.1 Пайплайн поиска релевантных документов

In [5]:
def retrieve_documents(query: str, top_k: int = 10) -> list:
    query_embedding = embedding_model.encode([query])
    
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    
    retrieved = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        retrieved.append({
            "text": doc,
            "metadata": meta,
            "similarity": 1 - dist
        })
    
    return retrieved

test_query = "право на образование"
retrieved = retrieve_documents(test_query, top_k=5)

print(f"Запрос: '{test_query}'")
print(f"Найдено документов: {len(retrieved)}\n")
for i, doc in enumerate(retrieved, 1):
    print(f"{i}. Статья {doc['metadata']['article_number']} "
          f"(схожесть: {doc['similarity']:.3f})")
    print(f"   Глава: {doc['metadata']['chapter_title']}")
    print(f"   {doc['text'][:120]}...\n")

Запрос: 'право на образование'
Найдено документов: 5

1. Статья 43 (схожесть: 0.693)
   Глава: ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА
   Статья 43
1. Каждый имеет право на образование.
2. Гарантируются общедоступность и бесплатность дошкольного, основного о...

2. Статья 44 (схожесть: 0.577)
   Глава: ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА
   Статья 44
1. Каждому гарантируется свобода литературного, художественного, научного, технического и других видов творчес...

3. Статья 2 (схожесть: 0.536)
   Глава: ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ
   Статья 2
Человек, его права и свободы являются высшей ценностью. Признание, соблюдение и защита прав и свобод человека и...

4. Статья 38 (схожесть: 0.491)
   Глава: ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА
   Статья 38
1. Материнство и детство, семья находятся под защитой государства.
2. Забота о детях, их воспитание - равное п...

5. Статья 19 (схожесть: 0.485)
   Глава: ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА
   Статья 19
1. Все равны перед законом и судом.

### 3.2 Реранкер — переранжирование результатов

In [6]:
from sentence_transformers import CrossEncoder

print("Загрузка реранкера...")
reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
print("Реранкер загружен: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")


def rerank_documents(query: str, documents: list, top_k: int = 5) -> list:
    if not documents:
        return []
    
    pairs = [(query, doc["text"]) for doc in documents]
    scores = reranker.predict(pairs)
    
    for doc, score in zip(documents, scores):
        doc["rerank_score"] = float(score)
    
    reranked = sorted(documents, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]


retrieved = retrieve_documents(test_query, top_k=10)
reranked = rerank_documents(test_query, retrieved, top_k=3)

print(f"\nЗапрос: '{test_query}'")
print("Топ-3 после реранжирования:")
for i, doc in enumerate(reranked, 1):
    print(f"\n{i}. Статья {doc['metadata']['article_number']} "
          f"(rerank_score: {doc['rerank_score']:.3f}, similarity: {doc['similarity']:.3f})")
    print(f"   {doc['text'][:200]}")

Загрузка реранкера...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6506.41it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Реранкер загружен: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1

Запрос: 'право на образование'
Топ-3 после реранжирования:

1. Статья 43 (rerank_score: 4.260, similarity: 0.693)
   Статья 43
1. Каждый имеет право на образование.
2. Гарантируются общедоступность и бесплатность дошкольного, основного общего и среднего профессионального образования в государственных или муниципальн

2. Статья 44 (rerank_score: -1.578, similarity: 0.577)
   Статья 44
1. Каждому гарантируется свобода литературного, художественного, научного, технического и других видов творчества, преподавания. Интеллектуальная собственность охраняется законом.
2. Каждый 

3. Статья 38 (rerank_score: -2.603, similarity: 0.491)
   Статья 38
1. Материнство и детство, семья находятся под защитой государства.
2. Забота о детях, их воспитание - равное право и обязанность родителей.
3. Трудоспособные дети, достигшие 18 лет, должны з


### 3.3 Системный промпт

In [8]:
import anthropic
import os

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

SYSTEM_PROMPT = """Ты — юридический ассистент, специализирующийся на Конституции Российской Федерации.
Твоя задача — помогать пользователям находить и интерпретировать правовые нормы, опираясь 
исключительно на предоставленный контекст из Конституции РФ.

ПРАВИЛА ОТВЕТА:
1. Отвечай ТОЛЬКО на основе предоставленного контекста — не придумывай и не дополняй информацию.
2. Если ответ найден в контексте — дай чёткий, структурированный ответ со ссылками на статьи.
3. Если контекст не содержит ответа — честно сообщи об этом.
4. Всегда указывай номера статей, на которые опираешься.
5. Цитируй ключевые формулировки из Конституции дословно, заключая их в кавычки.
6. Если вопрос неоднозначен — укажи разные аспекты.
7. Не давай юридических советов — только разъясняй нормы Конституции.
8. Отвечай на русском языке, грамотно и профессионально."""

### 3.4 Обращение к LLM

In [9]:
def build_user_message(query: str, context_docs: list) -> str:
    context_parts = []
    for i, doc in enumerate(context_docs, 1):
        meta = doc['metadata']
        context_parts.append(
            f"[Документ {i}]\n"
            f"Статья: {meta['article_number']}\n"
            f"Глава: {meta.get('chapter_number', '?')} — {meta.get('chapter_title', 'неизвестно')}\n"
            f"Текст:\n{doc['text']}"
        )
    
    context_str = "\n\n" + "-"*50 + "\n\n".join(context_parts)
    
    return (
        f"ВОПРОС ПОЛЬЗОВАТЕЛЯ:\n{query}\n\n"
        f"РЕЛЕВАНТНЫЕ СТАТЬИ ИЗ КОНСТИТУЦИИ РФ:\n{context_str}\n\n"
        f"Пожалуйста, ответь на вопрос, опираясь только на приведённые статьи."
    )


def ask_llm(query: str, top_k_retrieve: int = 10, top_k_rerank: int = 5) -> dict:
    retrieved = retrieve_documents(query, top_k=top_k_retrieve)
    
    reranked = rerank_documents(query, retrieved, top_k=top_k_rerank)
    
    user_message = build_user_message(query, reranked)
    
    if not ANTHROPIC_API_KEY:
        answer = (
            "API-ключ Anthropic не задан.\n\n"
            f"Найдено {len(reranked)} релевантных статей:\n"
            + "\n".join([
                f" * Статья {d['metadata']['article_number']} "
                f"(score: {d.get('rerank_score', 0):.2f})"
                for d in reranked
            ])
        )
    else:
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1500,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}]
        )
        answer = response.content[0].text
    
    return {
        "query": query,
        "answer": answer,
        "sources": [
            {
                "article": d["metadata"]["article_number"],
                "chapter": d["metadata"].get("chapter_title", ""),
                "rerank_score": round(d.get("rerank_score", 0), 3),
                "similarity": round(d["similarity"], 3),
                "text_preview": d["text"][:150] + "..."
            }
            for d in reranked
        ]
    }


result = ask_llm("Каковы права человека на свободу слова?")

print("=" * 70)
print(f"ВОПРОС: {result['query']}")
print("=" * 70)
print(f"\nОТВЕТ:\n{result['answer']}")
print("\n" + "-" * 70)
print("ИСТОЧНИКИ:")
for src in result['sources']:
    print(f"  • Статья {src['article']} | {src['chapter']} | score={src['rerank_score']}")

ВОПРОС: Каковы права человека на свободу слова?

ОТВЕТ:
API-ключ Anthropic не задан.

Найдено 5 релевантных статей:
 * Статья 29 (score: 1.12)
 * Статья 28 (score: -1.25)
 * Статья 23 (score: -1.65)
 * Статья 44 (score: -1.94)
 * Статья 18 (score: -2.37)

----------------------------------------------------------------------
ИСТОЧНИКИ:
  • Статья 29 | ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА | score=1.117
  • Статья 28 | ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА | score=-1.248
  • Статья 23 | ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА | score=-1.646
  • Статья 44 | ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА | score=-1.938
  • Статья 18 | ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА | score=-2.367


---
## Этап 4: Разработка API на FastAPI

In [10]:

api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import uvicorn

# Импортируем функции из основного модуля
# В реальном проекте эти функции выносятся в rag_pipeline.py

app = FastAPI(
    title="RAG API — Конституция РФ",
    description="API для поиска правовых статей из Конституции Российской Федерации",
    version="1.0.0"
)


class QueryRequest(BaseModel):
    query: str = Field(..., description="Вопрос пользователя", min_length=3)
    top_k: Optional[int] = Field(5, ge=1, le=10, description="Кол-во возвращаемых статей")


class SourceDoc(BaseModel):
    article: int
    chapter: str
    rerank_score: float
    similarity: float
    text_preview: str


class QueryResponse(BaseModel):
    query: str
    answer: str
    sources: List[SourceDoc]


@app.get("/", summary="Проверка работоспособности")
def health_check():
    return {"status": "ok", "service": "RAG — Конституция РФ"}


@app.post("/ask", response_model=QueryResponse, summary="Задать вопрос по Конституции РФ")
def ask_question(request: QueryRequest):
    """
    Принимает вопрос пользователя и возвращает ответ на основе Конституции РФ.
    
    - **query**: вопрос на русском языке
    - **top_k**: количество возвращаемых источников (1-10)
    """
    try:
        result = ask_llm(request.query, top_k_retrieve=15, top_k_rerank=request.top_k)
        return QueryResponse(
            query=result["query"],
            answer=result["answer"],
            sources=[SourceDoc(**src) for src in result["sources"]]
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/search", summary="Поиск статей без LLM")
def search_articles(q: str, top_k: int = 5):
    """Прямой поиск релевантных статей без обращения к LLM"""
    retrieved = retrieve_documents(q, top_k=top_k * 2)
    reranked = rerank_documents(q, retrieved, top_k=top_k)
    return {
        "query": q,
        "results": [
            {
                "article": d["metadata"]["article_number"],
                "chapter": d["metadata"].get("chapter_title", ""),
                "score": round(d.get("rerank_score", 0), 3),
                "text": d["text"]
            }
            for d in reranked
        ]
    }


# Запуск: uvicorn api:app --reload --port 8000
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("api.py", "w", encoding="utf-8") as f:
    f.write(api_code)

print(" Файл api.py создан")
print("\nДля запуска API выполните в терминале:")
print("  uvicorn api:app --reload --port 8000")
print("\nДокументация будет доступна по адресу:")
print("  http://localhost:8000/docs")

print("\n" + "="*60)
print("Эндпоинты API:")
print("  GET  /          — проверка работоспособности")
print("  POST /ask       — вопрос к RAG-системе")
print("  GET  /search    — поиск статей без LLM")
print("  GET  /docs      — Swagger UI документация")

 Файл api.py создан

Для запуска API выполните в терминале:
  uvicorn api:app --reload --port 8000

Документация будет доступна по адресу:
  http://localhost:8000/docs

Эндпоинты API:
  GET  /          — проверка работоспособности
  POST /ask       — вопрос к RAG-системе
  GET  /search    — поиск статей без LLM
  GET  /docs      — Swagger UI документация


---
## Этап 5: Web-интерфейс на Gradio

In [21]:
import gradio as gr

def format_sources(sources: list) -> str:
    if not sources:
        return "Источники не найдены"
    lines = ["**Использованные статьи:**"]
    for src in sources:
        lines.append(
            f"• **Статья {src['article']}** — {src['chapter']}  "
            f"*(релевантность: {src['rerank_score']:.2f})*"
        )
    return "\n".join(lines)

def chat_fn(message: str, history: list):
    if not message.strip():
        return history, "", ""
    result = ask_llm(message)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": result["answer"]})
    sources_text = format_sources(result["sources"])
    return history, "", sources_text

def clear_chat():
    return [], "", ""

examples = [
    "Какие права гарантированы каждому гражданину на труд?",
    "Что говорит Конституция о праве на образование?",
    "Каков порядок избрания Президента РФ?",
    "Что такое презумпция невиновности в Конституции?",
    "Какие права есть у задержанного гражданина?",
    "Можно ли лишить человека гражданства России?",
]

with gr.Blocks(title="RAG — Конституция РФ") as demo:
    gr.Markdown("""
    # RAG-система: Конституция Российской Федерации
    Задайте вопрос по Конституции РФ — система найдёт релевантные статьи и сформирует ответ.
    """)
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="Диалог",
                height=450
            )
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="Введите вопрос по Конституции РФ...",
                    label="Ваш вопрос",
                    scale=4
                )
                submit_btn = gr.Button("Спросить", variant="primary", scale=1)
            with gr.Row():
                clear_btn = gr.Button("Очистить историю", variant="secondary")
        with gr.Column(scale=1):
            sources_output = gr.Markdown(
                value="*Источники появятся после вашего вопроса*"
            )
    gr.Markdown("### Примеры вопросов:")
    gr.Examples(examples=examples, inputs=msg_input, label="")
    submit_btn.click(
        fn=chat_fn,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, msg_input, sources_output]
    )
    msg_input.submit(
        fn=chat_fn,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, msg_input, sources_output]
    )
    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot, msg_input, sources_output]
    )

print("Интерфейс Gradio настроен")
print("Для запуска выполните следующую ячейку")

Интерфейс Gradio настроен
Для запуска выполните следующую ячейку


In [23]:
demo.launch(share=False, server_port=7861)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


---
## Тестирование системы

In [24]:
test_questions = [
    "Что является высшей ценностью в Российской Федерации?",
    "Каковы требования к кандидату в Президенты?",
    "Что говорит Конституция о праве на жилище?",
    "Какова структура Федерального Собрания?",
]

print(" ТЕСТИРОВАНИЕ RAG-ПАЙПЛАЙНА")
print("=" * 70)

for i, question in enumerate(test_questions, 1):
    print(f"\nТест {i}: {question}")
    print("-" * 50)
    
    result = ask_llm(question)
    
    # Показываем только источники (ответ зависит от наличия API-ключа)
    print(f"Найдено источников: {len(result['sources'])}")
    for src in result['sources'][:3]:
        print(f"   Статья {src['article']} | score={src['rerank_score']:.3f} | "
              f"{src['chapter'][:40]}")
    
    print(f"\nОтвет системы:")
    print(result['answer'][:500])
    print()

print("\n Тестирование завершено")

 ТЕСТИРОВАНИЕ RAG-ПАЙПЛАЙНА

Тест 1: Что является высшей ценностью в Российской Федерации?
--------------------------------------------------
Найдено источников: 5
   Статья 7 | score=-2.700 | ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ
   Статья 15 | score=-2.762 | ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ
   Статья 3 | score=-3.229 | ОСНОВЫ КОНСТИТУЦИОННОГО СТРОЯ

Ответ системы:
API-ключ Anthropic не задан.

Найдено 5 релевантных статей:
 * Статья 7 (score: -2.70)
 * Статья 15 (score: -2.76)
 * Статья 3 (score: -3.23)
 * Статья 8 (score: -4.07)
 * Статья 13 (score: -4.18)


Тест 2: Каковы требования к кандидату в Президенты?
--------------------------------------------------
Найдено источников: 5
   Статья 82 | score=1.721 | ПРЕЗИДЕНТ РОССИЙСКОЙ ФЕДЕРАЦИИ
   Статья 81 | score=1.645 | ПРЕЗИДЕНТ РОССИЙСКОЙ ФЕДЕРАЦИИ
   Статья 32 | score=0.012 | ПРАВА И СВОБОДЫ ЧЕЛОВЕКА И ГРАЖДАНИНА

Ответ системы:
API-ключ Anthropic не задан.

Найдено 5 релевантных статей:
 * Статья 82 (score: 1.72)
 * Статья 81 (score: 1.65)
 

---
## Выводы

В данной лабораторной работе была разработана полноценная RAG-система для поиска правовых статей из Конституции Российской Федерации. 

### Реализованные этапы:

**Этап 1 — Технологический стек:**  
Выбраны открытые, бесплатные инструменты: `sentence-transformers` для эмбеддингов, `ChromaDB` как векторная БД, `FastAPI` для API и `Gradio` для web-интерфейса.

**Этап 2 — База знаний:**  
Текст Конституции РФ разбит на чанки по статьям с сохранением метаинформации (номер статьи, главы, раздела). Векторизовано 80+ статей и загружено в ChromaDB с косинусным расстоянием.

**Этап 3 — LLM и промпт-инжиниринг:**  
Реализован двухступенчатый поиск: первичный через `ChromaDB` (top-10), затем реранжирование через `cross-encoder` (top-5). Системный промпт содержит ролевую модель, инструкции против галлюцинаций и требование цитирования.

**Этап 4 — API:**  
Разработан FastAPI-сервис с эндпоинтами `/ask` (POST) и `/search` (GET), автодокументацией Swagger.

**Этап 5 — Web-интерфейс:**  
Реализован чат-интерфейс на Gradio с историей диалога, панелью источников и примерами вопросов.